# Corpus-regression sweep analysis

In [10]:
from src import get_repo_base
from src.experiments.corpus_regression.analysis import (
    CorpusRegressionAnalysisConfig,
    plot_methods_vs_epoch,
    plot_methods_vs_lookforward,
)
from src.experiments.corpus_regression.config import artifacts_dir

ARTIFACTS = artifacts_dir()

## Parameters

In [11]:
# === Parameters ===
NUM_SAMPLES: int = 50_000       # 100_000 or 500_000
GAUSSIAN_STDEV: float = 9.0      # 1.0 or 0.5
LABEL_TYPE: str = "token_id"   # "rademacher" or "token_id"

# Derived output path — namespaced by combo to avoid overwriting
_combo_slug = f"n{NUM_SAMPLES // 1000}k_sigma{GAUSSIAN_STDEV}"
if LABEL_TYPE != "rademacher":
    _combo_slug += f"_{LABEL_TYPE}"
WRITEUP_ASSETS = get_repo_base() / "writeup" / "assets" / "corpus-regression" / _combo_slug

## Supervised-learning

In [12]:
sl = CorpusRegressionAnalysisConfig.from_sl_sweep(
    artifacts_root=ARTIFACTS, num_samples=NUM_SAMPLES, label_type=LABEL_TYPE,
)
if sl is None:
    print("SL: no artifacts")
else:
    sl.describe("SL")
    display(
        sl.plot_vs_epoch(
            "corr",
            title="SL: per-epoch (corr)",
            show_seed_bar=True,
            save_path=WRITEUP_ASSETS / "sl_per_epoch_corr.html",
        )
    )
    display(
        sl.plot_vs_epoch(
            "mse",
            title="SL: per-epoch (mse)",
            show_seed_bar=True,
            save_path=WRITEUP_ASSETS / "sl_per_epoch_mse.html",
        )
    )
    display(
        sl.plot_vs_lookforward(
            title="SL: best-epoch vs num_lookforward_tokens",
            x_scale="uniform",
            show_seed_bar=True,
            save_path=WRITEUP_ASSETS / "sl_vs_lookforward.html",
        )
    )

SL                                   3 runs     1 groups   up to 3 seeds/group


## GRPO

In [13]:
grpo = CorpusRegressionAnalysisConfig.from_grpo_sweep(
    artifacts_root=ARTIFACTS, num_samples=NUM_SAMPLES, gaussian_stdev=GAUSSIAN_STDEV,
    label_type=LABEL_TYPE,
)
if grpo is None:
    print("GRPO: no artifacts")
else:
    grpo.describe("GRPO")
    display(
        grpo.plot_vs_epoch(
            "corr",
            title="GRPO: per-epoch (corr)",
            show_seed_bar=True,
            save_path=WRITEUP_ASSETS / "grpo_per_epoch_corr.html",
        )
    )
    display(
        grpo.plot_vs_epoch(
            "mse",
            title="GRPO: per-epoch (mse)",
            show_seed_bar=True,
            save_path=WRITEUP_ASSETS / "grpo_per_epoch_mse.html",
        )
    )
    display(
        grpo.plot_vs_lookforward(
            title="GRPO: best-epoch vs num_lookforward_tokens",
            x_scale="uniform",
            show_seed_bar=True,
            save_path=WRITEUP_ASSETS / "grpo_vs_lookforward.html",
        )
    )

GRPO                                 9 runs     3 groups   up to 3 seeds/group


## MaxRL (subtract-baseline + factorized)

In [14]:
maxrl_sf = CorpusRegressionAnalysisConfig.from_maxrl_sweep(
    artifacts_root=ARTIFACTS,
    subtract_baseline=True,
    use_factorized_likelihoods=True,
    num_samples=NUM_SAMPLES,
    gaussian_stdev=GAUSSIAN_STDEV,
    label_type=LABEL_TYPE,
)
if maxrl_sf is None:
    print("MaxRL (sub-baseline, factorized): no artifacts")
else:
    maxrl_sf.describe("MaxRL (sub-baseline, factorized)")
    display(
        maxrl_sf.plot_vs_epoch(
            "corr",
            title="MaxRL (sub-baseline, factorized): per-epoch (corr)",
            show_seed_bar=True,
            save_path=WRITEUP_ASSETS / "maxrl_sub_fact_per_epoch_corr.html",
        )
    )
    display(
        maxrl_sf.plot_vs_epoch(
            "mse",
            title="MaxRL (sub-baseline, factorized): per-epoch (mse)",
            show_seed_bar=True,
            save_path=WRITEUP_ASSETS / "maxrl_sub_fact_per_epoch_mse.html",
        )
    )
    display(
        maxrl_sf.plot_vs_lookforward(
            title="MaxRL (sub-baseline, factorized): best-epoch vs num_lookforward_tokens",
            x_scale="uniform",
            show_seed_bar=True,
            save_path=WRITEUP_ASSETS / "maxrl_sub_fact_vs_lookforward.html",
        )
    )

MaxRL (sub-baseline, factorized)     8 runs     3 groups   up to 3 seeds/group


### Other MaxRL ablations

Uncomment to inspect the other (`subtract_baseline`, `use_factorized_likelihoods`) combinations if we run these sweeps

In [15]:
# for sub, fact in [(True, False), (False, True), (False, False)]:
#     cfg = CorpusRegressionAnalysisConfig.from_maxrl_sweep(
#         artifacts_root=ARTIFACTS,
#         subtract_baseline=sub,
#         use_factorized_likelihoods=fact,
#         num_samples=NUM_SAMPLES,
#         gaussian_stdev=GAUSSIAN_STDEV,
#     )
#     label = f"MaxRL (sub={sub}, fact={fact})"
#     if cfg is None:
#         print(f"{label}: no artifacts")
#         continue
#     cfg.describe(label)
#     display(cfg.plot_vs_epoch("corr", title=f"{label}: per-epoch (corr)", show_seed_bar=True))
#     display(cfg.plot_vs_lookforward(title=f"{label}: best-epoch vs num_lookforward_tokens", x_scale="uniform", show_seed_bar=True))

## RLOO (factorized)

In [16]:
rloo_f = CorpusRegressionAnalysisConfig.from_rloo_sweep(
    artifacts_root=ARTIFACTS,
    factorized=True,
    num_samples=NUM_SAMPLES,
    gaussian_stdev=GAUSSIAN_STDEV,
    label_type=LABEL_TYPE,
)
if rloo_f is None:
    print("RLOO (factorized): no artifacts")
else:
    rloo_f.describe("RLOO (factorized)")
    display(
        rloo_f.plot_vs_epoch(
            "corr",
            title="RLOO (factorized): per-epoch (corr)",
            show_seed_bar=True,
            save_path=WRITEUP_ASSETS / "rloo_factorized_per_epoch_corr.html",
        )
    )
    display(
        rloo_f.plot_vs_epoch(
            "mse",
            title="RLOO (factorized): per-epoch (mse)",
            show_seed_bar=True,
            save_path=WRITEUP_ASSETS / "rloo_factorized_per_epoch_mse.html",
        )
    )
    display(
        rloo_f.plot_vs_lookforward(
            title="RLOO (factorized): best-epoch vs num_lookforward_tokens",
            x_scale="uniform",
            show_seed_bar=True,
            save_path=WRITEUP_ASSETS / "rloo_factorized_vs_lookforward.html",
        )
    )

RLOO (factorized)                    9 runs     3 groups   up to 3 seeds/group


## Cross-method comparison

In [17]:
if any(c is not None for c in (sl, grpo, maxrl_sf, rloo_f)):
    display(
        plot_methods_vs_lookforward(
            sl=sl,
            grpo=grpo,
            maxrl=maxrl_sf,
            rloo=rloo_f,
            title="Methods: best-epoch corr vs num_lookforward_tokens",
            x_scale="uniform",
            save_path=WRITEUP_ASSETS / "methods_vs_lookforward.html",
        )
    )
    display(
        plot_methods_vs_lookforward(
            sl=sl,
            grpo=grpo,
            maxrl=maxrl_sf,
            rloo=rloo_f,
            metric="mse",
            title="Methods: best-epoch MSE vs num_lookforward_tokens",
            x_scale="uniform",
            save_path=WRITEUP_ASSETS / "methods_vs_lookforward_mse.html",
        )
    )
else:
    print("No artifacts for any method.")

### Per-epoch training curves (look=1, all methods)

In [18]:
if any(c is not None for c in (sl, grpo, maxrl_sf, rloo_f)):
    display(
        plot_methods_vs_epoch(
            sl=sl,
            grpo=grpo,
            maxrl=maxrl_sf,
            rloo=rloo_f,
            num_lookforward_tokens=1,
            metric="corr",
            show_seed_bar=True,
            title="Methods: per-epoch corr (look=1)",
            save_path=WRITEUP_ASSETS / "methods_vs_epoch_look1_corr.html",
        )
    )
    display(
        plot_methods_vs_epoch(
            sl=sl,
            grpo=grpo,
            maxrl=maxrl_sf,
            rloo=rloo_f,
            num_lookforward_tokens=1,
            metric="mse",
            show_seed_bar=True,
            title="Methods: per-epoch MSE (look=1)",
            save_path=WRITEUP_ASSETS / "methods_vs_epoch_look1_mse.html",
        )
    )
else:
    print("No artifacts for any method.")